In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
!pip install pandas nltk scikit-learn

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
 #MODULE 1: EMAIL PREPROCESSING

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
df = pd.read_csv("train.csv")

if "subject" in df.columns and "body" in df.columns:
    df["text"] = df["subject"].fillna('') + " " + df["body"].fillna('')

# If dataset already has text column, it will use it directly
df["text"] = df["text"].fillna("")


def clean_email(text):
    text = str(text)

    # 1. Remove Email Threads
    text = re.split(r"From:|On .* wrote:|-----Original Message-----", text)[0]

    # 2. Remove Signatures
    text = re.split(r"Regards|Best regards|Sincerely|Thanks & regards", text, flags=re.IGNORECASE)[0]

    # 3. Remove Disclaimers
    text = re.split(r"Disclaimer:|This email.*confidential", text, flags=re.IGNORECASE)[0]

    # 4. Normalize Spacing
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    # 5. Remove Stopwords
    words = text.split()
    words = [w for w in words if w.lower() not in stop_words]
    text = " ".join(words)

    # 6. Limit Maximum Length (200 words)
    text = " ".join(text.split()[:200])

    return text

# Apply cleaning
df["clean_text"] = df["text"].apply(clean_email)

# Show cleaned data
df[["text", "clean_text"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,clean_text
0,Anniversary Special: Buy one get one free As o...,Anniversary Special: Buy one get one free loya...
1,Your Amazon was used on new device Your $5000 ...,Amazon used new device $5000 refund processed....
2,"Re: Your Google inquiry Hi, following up about...","Re: Google inquiry Hi, following Google applic..."
3,Digital Ritual Experience Creation Cross-cultu...,Digital Ritual Experience Creation Cross-cultu...
4,"Your post was moved to ""Programming Help"" Tren...","post moved ""Programming Help"" Trending: ""cooki..."


In [24]:
#MODULE 2:Email Categorization Engine

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
X = df["clean_text"]
y = df["category"]
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9865491651205937

Classification Report:

              precision    recall  f1-score   support

       forum       0.98      0.98      0.98       360
  promotions       0.99      0.99      0.99       355
social_media       0.98      0.98      0.98       352
        spam       0.99      0.99      0.99       355
     updates       0.98      0.99      0.98       377
 verify_code       0.99      0.99      0.99       357

    accuracy                           0.99      2156
   macro avg       0.99      0.99      0.99      2156
weighted avg       0.99      0.99      0.99      2156

